# Milestone 5: GraphSAGE Recommender

This notebook trains a GraphSAGE model for user-movie recommendation using the processed files from earlier milestones.

Pipeline:
1. Load train positives and candidate sets
2. Build a bipartite training graph from user-movie interactions
3. Train GraphSAGE with negative sampling
4. Evaluate with Recall@10 and NDCG@10
5. Save scored validation and test candidates

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import importlib
import random
import subprocess
import sys
from pathlib import Path


def ensure_package(pip_name, import_name=None):
    """Install a package only if it is missing.

    Why: keeps the notebook reproducible on fresh environments (local/Colab).
    """
    module_name = import_name or pip_name
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])


# Core dependencies used across the full pipeline.
# Why these: numpy/pandas for data prep, torch + pyg for GraphSAGE training.
ensure_package('numpy')
ensure_package('pandas')
ensure_package('torch')
ensure_package('torch-geometric', 'torch_geometric')

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

# Fix random seeds so results are more stable across runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use GPU if available, otherwise CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [4]:
# Resolve project root automatically.
# Why: lets the same notebook run both in local VS Code and in Colab.
CANDIDATE_ROOTS = [
    Path('.'),
    Path('/content/drive/MyDrive/MLG Project/Project'),
]

PROJECT_ROOT = None
for root in CANDIDATE_ROOTS:
    if (root / 'data' / 'processed').exists():
        PROJECT_ROOT = root
        break

# Safe fallback so relative paths still work if the folder is not detected.
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.')

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED

PosixPath('/content/drive/MyDrive/MLG Project/Project/data/processed')

In [5]:
# Load processed artifacts from previous milestones.
# Why: train_pos builds the graph; candidates are used for ranking evaluation.
train_pos = pd.read_csv(DATA_PROCESSED / 'train_pos.csv')
val_candidates = pd.read_csv(DATA_PROCESSED / 'val_candidates.csv')
test_candidates = pd.read_csv(DATA_PROCESSED / 'test_candidates.csv')

print('Train positives:', len(train_pos))
print('Validation candidates:', len(val_candidates))
print('Test candidates:', len(test_candidates))

# Quick peek to confirm schema looks right.
train_pos.head()

Train positives: 47746
Validation candidates: 19278
Test candidates: 23256


,userId,movieId,timestamp
0,429,150,828124615
1,429,161,828124615
2,429,351,828124615
3,429,595,828124615
4,429,421,828124615


In [6]:
# Map raw IDs (userId/movieId) to contiguous integer indices.
# Why: GNN layers and embedding tables require index-based node IDs.
users = sorted(train_pos['userId'].unique())
movies = sorted(train_pos['movieId'].unique())

num_users = len(users)
num_movies = len(movies)
num_nodes = num_users + num_movies

# Users occupy [0, num_users-1], movies occupy [num_users, ...].
# Why offset movie indices: prevents collisions between user and movie nodes.
user_to_idx = {u: i for i, u in enumerate(users)}
movie_to_idx = {m: i + num_users for i, m in enumerate(movies)}

train_edges = train_pos[['userId', 'movieId']].drop_duplicates().copy()
train_edges['u_idx'] = train_edges['userId'].map(user_to_idx)
train_edges['m_idx'] = train_edges['movieId'].map(movie_to_idx)

# Build an undirected bipartite graph in COO format.
# Why undirected: message passing should flow user->movie and movie->user.
src = torch.tensor(train_edges['u_idx'].values, dtype=torch.long)
dst = torch.tensor(train_edges['m_idx'].values, dtype=torch.long)
edge_index = torch.stack([
    torch.cat([src, dst]),
    torch.cat([dst, src])
], dim=0).to(DEVICE)

# For negative sampling: movies each user already interacted with.
user_pos_movies = train_edges.groupby('u_idx')['m_idx'].apply(set).to_dict()
all_movie_nodes = np.array(list(movie_to_idx.values()))

print(f'Users: {num_users}, Movies: {num_movies}, Total nodes: {num_nodes}')
print(f'Undirected edges: {edge_index.size(1)}')

Users: 609, Movies: 6298, Total nodes: 6907
Undirected edges: 95492


In [7]:
class GraphSAGERecommender(nn.Module):
    """Two-layer GraphSAGE encoder + dot-product scorer.

    Why this design:
    - Embedding table gives each node a learnable base vector.
    - GraphSAGE aggregates neighbor information (collaborative signal).
    - Dot product between user/movie embeddings gives compatibility score.
    """

    def __init__(self, num_nodes, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = dropout

    def encode(self, edge_index):
        # Start from learned node embeddings, then run 2 graph conv layers.
        x = self.embedding.weight
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def score_pairs(self, z, u_idx, m_idx):
        # Dot product is a standard implicit-feedback scoring function.
        return (z[u_idx] * z[m_idx]).sum(dim=-1)


def sample_neg_movies(user_indices, user_pos_movies, all_movie_nodes):
    """Sample one unseen movie per user for pairwise-style training.

    Why: implicit recommendation needs negatives; we treat unseen interactions
    as negative examples.
    """
    neg_movies = []
    for u in user_indices:
        seen = user_pos_movies.get(int(u), set())
        while True:
            m_neg = int(np.random.choice(all_movie_nodes))
            if m_neg not in seen:
                neg_movies.append(m_neg)
                break
    return torch.tensor(neg_movies, dtype=torch.long, device=DEVICE)


def to_index_tensors(df):
    """Convert raw IDs to model indices and drop unknown entities.

    Why dropping unknowns: inference only works for users/movies that exist in
    the training graph embedding space.
    """
    u = df['userId'].map(user_to_idx)
    m = df['movieId'].map(movie_to_idx)
    valid = u.notna() & m.notna()
    df_valid = df.loc[valid].copy()

    u_idx = torch.tensor(u[valid].astype(int).values, dtype=torch.long, device=DEVICE)
    m_idx = torch.tensor(m[valid].astype(int).values, dtype=torch.long, device=DEVICE)
    labels = torch.tensor(df_valid['label'].values, dtype=torch.float32, device=DEVICE) if 'label' in df_valid.columns else None
    return df_valid, u_idx, m_idx, labels

In [8]:
# Positive training interactions (user, movie) from observed edges.
train_u = torch.tensor(train_edges['u_idx'].values, dtype=torch.long, device=DEVICE)
train_m = torch.tensor(train_edges['m_idx'].values, dtype=torch.long, device=DEVICE)

# Try multiple learning rates and train each run for at least 100 epochs.
learning_rates = [1e-4, 5e-4, 1e-3, 3e-3]
EPOCHS = 100
torch.manual_seed(42)
np.random.seed(42)


def train_one_model(lr, epochs=100):
    model = GraphSAGERecommender(num_nodes=num_nodes, hidden_dim=64, dropout=0.2).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    final_loss = None
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        # Encode full graph once per epoch.
        z = model.encode(edge_index)

        # Positive scores for observed interactions.
        pos_scores = model.score_pairs(z, train_u, train_m)

        # Negative scores for sampled unobserved interactions.
        neg_m = sample_neg_movies(train_u.cpu().numpy(), user_pos_movies, all_movie_nodes)
        neg_scores = model.score_pairs(z, train_u, neg_m)

        # Binary objective: positives -> 1, negatives -> 0.
        logits = torch.cat([pos_scores, neg_scores], dim=0)
        labels = torch.cat([
            torch.ones_like(pos_scores),
            torch.zeros_like(neg_scores)
        ], dim=0)

        loss = F.binary_cross_entropy_with_logits(logits, labels)
        loss.backward()
        optimizer.step()

        final_loss = float(loss.item())
        if epoch % 20 == 0 or epoch == 1:
            print(f'lr={lr:.0e} | Epoch {epoch:03d}/{epochs} | Loss: {final_loss:.4f}')

    return model, final_loss


trained_models = {}
for lr in learning_rates:
    print(f'\nTraining GraphSAGE with learning rate {lr:.0e}')
    model_lr, last_loss = train_one_model(lr, epochs=EPOCHS)
    trained_models[lr] = {'model': model_lr, 'final_loss': last_loss}

print('\nFinished training all learning-rate runs.')


Training GraphSAGE with learning rate 1e-04
lr=1e-04 | Epoch 001/100 | Loss: 1.8927
lr=1e-04 | Epoch 020/100 | Loss: 1.2509
lr=1e-04 | Epoch 040/100 | Loss: 0.9746
lr=1e-04 | Epoch 060/100 | Loss: 0.8746
lr=1e-04 | Epoch 080/100 | Loss: 0.8086
lr=1e-04 | Epoch 100/100 | Loss: 0.7850

Training GraphSAGE with learning rate 5e-04
lr=5e-04 | Epoch 001/100 | Loss: 1.3166
lr=5e-04 | Epoch 020/100 | Loss: 0.7258
lr=5e-04 | Epoch 040/100 | Loss: 0.6895
lr=5e-04 | Epoch 060/100 | Loss: 0.6583
lr=5e-04 | Epoch 080/100 | Loss: 0.6104
lr=5e-04 | Epoch 100/100 | Loss: 0.5577

Training GraphSAGE with learning rate 1e-03
lr=1e-03 | Epoch 001/100 | Loss: 1.5315
lr=1e-03 | Epoch 020/100 | Loss: 0.7055
lr=1e-03 | Epoch 040/100 | Loss: 0.6345
lr=1e-03 | Epoch 060/100 | Loss: 0.5607
lr=1e-03 | Epoch 080/100 | Loss: 0.4855
lr=1e-03 | Epoch 100/100 | Loss: 0.4373

Training GraphSAGE with learning rate 3e-03
lr=3e-03 | Epoch 001/100 | Loss: 1.5677
lr=3e-03 | Epoch 020/100 | Loss: 0.6112
lr=3e-03 | Epoch 040

In [9]:
@torch.no_grad()
def score_candidates(candidates_df, model, edge_index):
    """Score each (user, movie) candidate pair with the trained model."""
    model.eval()
    z = model.encode(edge_index)

    df_valid, u_idx, m_idx, _ = to_index_tensors(candidates_df)
    scores = model.score_pairs(z, u_idx, m_idx).cpu().numpy()

    out = df_valid.copy()
    out['graphsage_score'] = scores
    return out


def recall_ndcg_at_k(df, score_col='graphsage_score', k=10):
    """Compute ranking metrics per user and average.

    Why Recall@K: measures ability to retrieve relevant items.
    Why NDCG@K: also rewards putting relevant items near top ranks.
    """
    recalls, ndcgs = [], []
    for _, g in df.groupby('userId'):
        g = g.sort_values(score_col, ascending=False)
        topk = g.head(k)

        labels = topk['label'].values
        total_pos = int(g['label'].sum())
        if total_pos == 0:
            continue

        hits = int(labels.sum())
        recall = hits / total_pos

        dcg = 0.0
        for i, rel in enumerate(labels, start=1):
            if rel > 0:
                dcg += 1.0 / np.log2(i + 1)

        ideal_hits = min(total_pos, k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0

        recalls.append(recall)
        ndcgs.append(ndcg)

    return float(np.mean(recalls)), float(np.mean(ndcgs))


results = []
best_lr = None
best_val_recall10 = -1.0
best_model = None
best_val_scored = None
best_test_scored = None

for lr, payload in trained_models.items():
    model_candidate = payload['model']
    val_scored_lr = score_candidates(val_candidates, model_candidate, edge_index)
    test_scored_lr = score_candidates(test_candidates, model_candidate, edge_index)

    val_recall10, val_ndcg10 = recall_ndcg_at_k(val_scored_lr, k=10)
    test_recall10, test_ndcg10 = recall_ndcg_at_k(test_scored_lr, k=10)

    results.append({
        'lr': lr,
        'final_loss': payload['final_loss'],
        'val_recall10': val_recall10,
        'val_ndcg10': val_ndcg10,
        'test_recall10': test_recall10,
        'test_ndcg10': test_ndcg10,
    })

    print(
        f"lr={lr:.0e} | "
        f"Val Recall@10: {val_recall10:.4f}, NDCG@10: {val_ndcg10:.4f} | "
        f"Test Recall@10: {test_recall10:.4f}, NDCG@10: {test_ndcg10:.4f}"
    )

    if val_recall10 > best_val_recall10:
        best_val_recall10 = val_recall10
        best_lr = lr
        best_model = model_candidate
        best_val_scored = val_scored_lr
        best_test_scored = test_scored_lr

results_df = pd.DataFrame(results).sort_values('val_recall10', ascending=False).reset_index(drop=True)
print('\nLearning-rate sweep summary (sorted by validation Recall@10):')
print(results_df)

# Use the best validation-recall model for downstream export and analysis.
model = best_model
val_scored = best_val_scored
test_scored = best_test_scored

print(f"\nSelected best learning rate: {best_lr:.0e}")
print(f"Best validation Recall@10: {best_val_recall10:.4f}")

lr=1e-04 | Val Recall@10: 0.0627, NDCG@10: 0.0223 | Test Recall@10: 0.0046, NDCG@10: 0.0049
lr=5e-04 | Val Recall@10: 0.1176, NDCG@10: 0.0504 | Test Recall@10: 0.0122, NDCG@10: 0.0147
lr=1e-03 | Val Recall@10: 0.1916, NDCG@10: 0.1928 | Test Recall@10: 0.1245, NDCG@10: 0.1728
lr=3e-03 | Val Recall@10: 0.2816, NDCG@10: 0.1816 | Test Recall@10: 0.1380, NDCG@10: 0.2274

Learning-rate sweep summary (sorted by validation Recall@10):
       lr  final_loss  val_recall10  val_ndcg10  test_recall10  test_ndcg10
0  0.0030    0.308880      0.281593    0.181574       0.138037     0.227419
1  0.0010    0.437285      0.191551    0.192836       0.124520     0.172813
2  0.0005    0.557698      0.117647    0.050365       0.012211     0.014738
3  0.0001    0.784968      0.062745    0.022319       0.004630     0.004897

Selected best learning rate: 3e-03
Best validation Recall@10: 0.2816


In [10]:
# Targeted recall tuning for GraphSAGE (aim: validation Recall@10 >= 0.50).

def train_tuned_model(lr, hidden_dim=128, dropout=0.1, epochs=220, neg_ratio=3, weight_decay=1e-6):
    model = GraphSAGERecommender(num_nodes=num_nodes, hidden_dim=hidden_dim, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        z = model.encode(edge_index)
        pos_scores = model.score_pairs(z, train_u, train_m)

        # Use multiple sampled negatives per positive to improve ranking discrimination.
        neg_u = train_u.repeat_interleave(neg_ratio)
        neg_m = sample_neg_movies(neg_u.detach().cpu().numpy(), user_pos_movies, all_movie_nodes)
        neg_scores = model.score_pairs(z, neg_u, neg_m)

        logits = torch.cat([pos_scores, neg_scores], dim=0)
        labels = torch.cat([
            torch.ones_like(pos_scores),
            torch.zeros_like(neg_scores)
        ], dim=0)

        # Counter class imbalance from extra negatives.
        pos_weight = torch.tensor([float(neg_ratio)], device=DEVICE)
        loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)
        loss.backward()
        optimizer.step()

        if epoch % 40 == 0 or epoch == 1:
            print(f"cfg(lr={lr:.0e}, h={hidden_dim}, d={dropout}, neg={neg_ratio}) | Epoch {epoch:03d}/{epochs} | Loss: {loss.item():.4f}")

    return model


search_space = [
    {"lr": 1e-3, "hidden_dim": 128, "dropout": 0.10, "epochs": 220, "neg_ratio": 3},
    {"lr": 3e-3, "hidden_dim": 128, "dropout": 0.10, "epochs": 220, "neg_ratio": 3},
    {"lr": 5e-3, "hidden_dim": 128, "dropout": 0.05, "epochs": 220, "neg_ratio": 3},
    {"lr": 3e-3, "hidden_dim": 192, "dropout": 0.15, "epochs": 240, "neg_ratio": 4},
]

tuned_results = []
target_recall10 = 0.50
best_tuned_recall10 = best_val_recall10
best_tuned_model = model
best_tuned_val_scored = val_scored
best_tuned_test_scored = test_scored
best_tuned_cfg = {"lr": best_lr, "hidden_dim": 64, "dropout": 0.2, "epochs": EPOCHS, "neg_ratio": 1}

for cfg in search_space:
    print(f"\nTraining tuned config: {cfg}")
    model_cfg = train_tuned_model(**cfg)

    val_scored_cfg = score_candidates(val_candidates, model_cfg, edge_index)
    test_scored_cfg = score_candidates(test_candidates, model_cfg, edge_index)

    val_recall10, val_ndcg10 = recall_ndcg_at_k(val_scored_cfg, k=10)
    test_recall10, test_ndcg10 = recall_ndcg_at_k(test_scored_cfg, k=10)

    row = {
        **cfg,
        "val_recall10": val_recall10,
        "val_ndcg10": val_ndcg10,
        "test_recall10": test_recall10,
        "test_ndcg10": test_ndcg10,
    }
    tuned_results.append(row)

    print(
        f"Tuned result | Val Recall@10: {val_recall10:.4f}, NDCG@10: {val_ndcg10:.4f} | "
        f"Test Recall@10: {test_recall10:.4f}, NDCG@10: {test_ndcg10:.4f}"
    )

    if val_recall10 > best_tuned_recall10:
        best_tuned_recall10 = val_recall10
        best_tuned_model = model_cfg
        best_tuned_val_scored = val_scored_cfg
        best_tuned_test_scored = test_scored_cfg
        best_tuned_cfg = cfg

    if val_recall10 >= target_recall10:
        print(f"Reached target validation Recall@10 >= {target_recall10:.2f}. Early stopping search.")
        break

tuned_results_df = pd.DataFrame(tuned_results).sort_values("val_recall10", ascending=False).reset_index(drop=True)
print("\nTuned sweep summary (sorted by validation Recall@10):")
print(tuned_results_df)

if best_tuned_recall10 > best_val_recall10:
    model = best_tuned_model
    val_scored = best_tuned_val_scored
    test_scored = best_tuned_test_scored
    best_val_recall10 = best_tuned_recall10
    print(f"\nUpdated selected model to tuned config: {best_tuned_cfg}")
else:
    print("\nNo tuned config outperformed baseline best model.")

print(f"Best validation Recall@10 after tuning: {best_val_recall10:.4f}")


Training tuned config: {'lr': 0.001, 'hidden_dim': 128, 'dropout': 0.1, 'epochs': 220, 'neg_ratio': 3}
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 001/220 | Loss: 4.9676
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 040/220 | Loss: 0.7926
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 080/220 | Loss: 0.5184
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 120/220 | Loss: 0.4153
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 160/220 | Loss: 0.3529
cfg(lr=1e-03, h=128, d=0.1, neg=3) | Epoch 200/220 | Loss: 0.3128
Tuned result | Val Recall@10: 0.2934, NDCG@10: 0.3022 | Test Recall@10: 0.1861, NDCG@10: 0.2535

Training tuned config: {'lr': 0.003, 'hidden_dim': 128, 'dropout': 0.1, 'epochs': 220, 'neg_ratio': 3}
cfg(lr=3e-03, h=128, d=0.1, neg=3) | Epoch 001/220 | Loss: 3.1961
cfg(lr=3e-03, h=128, d=0.1, neg=3) | Epoch 040/220 | Loss: 0.5563
cfg(lr=3e-03, h=128, d=0.1, neg=3) | Epoch 080/220 | Loss: 0.3566
cfg(lr=3e-03, h=128, d=0.1, neg=3) | Epoch 120/220 | Loss: 0.2737
cfg(lr=3e-03, h=128, d=0.1, neg=3) |

In [11]:
# Threshold calibration for binary recall target (recall >= 0.50).
# This complements ranking Recall@10 and is useful when you need minimum retrieval sensitivity.

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def binary_metrics_at_threshold(df, thr, score_col='graphsage_score'):
    probs = sigmoid_np(df[score_col].values)
    y_true = df['label'].values.astype(int)
    y_pred = (probs >= thr).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return recall, precision, tp, fp, fn


threshold_grid = np.linspace(0.01, 0.99, 99)
TARGET_RECALL = 0.50

feasible = []
for thr in threshold_grid:
    r, p, tp, fp, fn = binary_metrics_at_threshold(val_scored, thr)
    if r >= TARGET_RECALL:
        feasible.append((thr, r, p, tp, fp, fn))

if feasible:
    # Among thresholds meeting recall target, choose the one with best precision.
    best_thr, val_r, val_p, val_tp, val_fp, val_fn = sorted(feasible, key=lambda x: x[2], reverse=True)[0]
    test_r, test_p, test_tp, test_fp, test_fn = binary_metrics_at_threshold(test_scored, best_thr)

    print(f"Selected threshold: {best_thr:.2f}")
    print(f"Validation recall: {val_r:.4f}, precision: {val_p:.4f} | TP={val_tp}, FP={val_fp}, FN={val_fn}")
    print(f"Test recall:       {test_r:.4f}, precision: {test_p:.4f} | TP={test_tp}, FP={test_fp}, FN={test_fn}")
else:
    print("No threshold in the grid reached validation recall >= 0.50.")

Selected threshold: 0.50
Validation recall: 0.5000, precision: 0.1662 | TP=189, FP=948, FN=189
Test recall:       0.5088, precision: 0.1785 | TP=232, FP=1068, FN=224


In [12]:
# Persist scored candidate files for reporting/comparison with baselines.
val_out_path = DATA_PROCESSED / 'val_candidates_graphsage_scored.csv'
test_out_path = DATA_PROCESSED / 'test_candidates_graphsage_scored.csv'

best_val_scored.to_csv(val_out_path, index=False)
best_test_scored.to_csv(test_out_path, index=False)

print('Saved:')
print(val_out_path)
print(test_out_path)

# Show an example user ranking to inspect recommendation quality manually.
example_user = val_scored['userId'].iloc[0]
print(f'\nTop 10 recommendations for user {example_user} (validation candidates):')
val_scored[val_scored['userId'] == example_user].sort_values('graphsage_score', ascending=False).head(10)

Saved:
/content/drive/MyDrive/MLG Project/Project/data/processed/val_candidates_graphsage_scored.csv
/content/drive/MyDrive/MLG Project/Project/data/processed/test_candidates_graphsage_scored.csv

Top 10 recommendations for user 249 (validation candidates):


,userId,movieId,label,split,graphsage_score
4284,249,68358,1,val,6.986344
16370,249,51255,0,val,6.016675
9098,249,5010,0,val,5.836088
7446,249,4776,1,val,5.775072
18435,249,54286,0,val,5.510557
9766,249,68954,0,val,5.242465
18402,249,54997,0,val,4.749329
7408,249,5574,0,val,4.721100
7342,249,5064,0,val,4.660594
16822,249,5064,0,val,4.660594
